# 层和块（Layers and Blocks）

## 1. 神经网络的基本组成

神经网络主要由层（Layer）和块（Block）组成。

每个层通常具有以下特点：

- **接收输入**：处理上一层传递的数据。
- **产生输出**：将计算结果传递给下一层。
- **管理参数**：部分层包含可训练的权重和偏置。

例如，多层感知机（MLP）由多个全连接层和激活函数组成。

## 2. 什么是块（Block）？

**块是神经网络中的可复用组件，可以表示单个层、多个层的组合，甚至整个模型。**

例如：

- 单个全连接层可以看作一个块。
- 全连接层与 ReLU 激活函数可以组成一个块。
- 多个块可以进一步组合成完整的神经网络。

### 层与块的组合关系

![多个层被组合成块，形成更大的模型](../img/blocks.svg)

**核心思想：将复杂的神经网络拆分为多个可复用的模块，再组合成更大的模型。**

例如，ResNet 等深度神经网络就是通过重复组合不同的网络块来构建复杂结构。

## 3. PyTorch 中的块

在 PyTorch 中，块通常通过继承 `nn.Module` 类实现。

一个自定义块主要包含：

1. `__init__()`：定义网络层和需要管理的参数。
2. `forward()`：定义输入数据如何经过各层得到输出。

反向传播通常由 PyTorch 的自动求导机制完成，无需手动实现。

## 4. 使用 Sequential 构建模型

```python
import torch
from torch import nn

# 构建单隐藏层多层感知机
net = nn.Sequential(
    nn.Linear(20, 256),  # 输入层到隐藏层
    nn.ReLU(),           # 激活函数
    nn.Linear(256, 10)   # 隐藏层到输出层
)

X = torch.rand(2, 20)  # 生成2个样本，每个20个特征
net(X)                 # 执行前向传播
```

其中：

- `nn.Linear(20, 256)`：将20个特征映射为256个隐藏单元。
- `nn.ReLU()`：引入非线性。
- `nn.Linear(256, 10)`：输出10个类别的预测分数。
- `nn.Sequential()`：将多个层按顺序组合成一个块。

## 5. 总结

- **层（Layer）**：神经网络的基本计算单元。
- **块（Block）**：由一个或多个层组成的可复用组件。
- **nn.Module**：PyTorch 构建自定义层和块的基础类。
- **nn.Sequential**：按顺序组织多个神经网络层。
- **模块化设计**：通过组合和复用块，简化复杂神经网络的构建。

**核心思想：层组成块，块可以继续组合成更大的块，最终形成完整的神经网络。**

In [1]:
import torch  # PyTorch深度学习框架
from torch import nn  # 神经网络模块，用于构建网络层
from torch.nn import functional as F  # 神经网络函数接口（当前代码未使用）

# 使用Sequential构建单隐藏层多层感知机（MLP）
net = nn.Sequential(
    nn.Linear(20, 256),  # 全连接层：20个输入特征 → 256个隐藏单元
    nn.ReLU(),           # ReLU激活函数，引入非线性
    nn.Linear(256, 10)   # 输出层：256个隐藏单元 → 10个输出
)

# 生成2个样本，每个样本包含20个随机特征
X = torch.rand(2, 20)

net(X)  # 执行前向传播，输出形状为(2, 10)

tensor([[ 1.0066e-01,  1.7206e-01, -2.0513e-02, -2.4173e-01,  1.7790e-01,
          7.1500e-02,  5.6810e-02,  1.4393e-01, -2.4624e-01,  1.2912e-01],
        [ 1.8465e-02,  5.7015e-02, -2.1268e-04, -1.2388e-01,  1.0251e-01,
          1.5944e-01,  4.5121e-03,  1.1005e-01, -9.5477e-02,  1.5255e-01]],
       grad_fn=<AddmmBackward0>)

在这个例子中，我们通过实例化`nn.Sequential`来构建我们的模型，
层的执行顺序是作为参数传递的。
简而言之，(**`nn.Sequential`定义了一种特殊的`Module`**)，
即在PyTorch中表示一个块的类，
它维护了一个由`Module`组成的有序列表。
注意，两个全连接层都是`Linear`类的实例，
`Linear`类本身就是`Module`的子类。
另外，到目前为止，我们一直在通过`net(X)`调用我们的模型来获得模型的输出。
这实际上是`net.__call__(X)`的简写。
这个前向传播函数非常简单：
它将列表中的每个块连接在一起，将每个块的输出作为下一个块的输入。


## **自定义块**

要想直观地了解块是如何工作的，最简单的方法就是自己实现一个。
在实现我们自定义块之前，我们简要总结一下每个块必须提供的基本功能。


1. 将输入数据作为其前向传播函数的参数。
1. 通过前向传播函数来生成输出。请注意，输出的形状可能与输入的形状不同。例如，我们上面模型中的第一个全连接的层接收一个20维的输入，但是返回一个维度为256的输出。
1. 计算其输出关于输入的梯度，可通过其反向传播函数进行访问。通常这是自动发生的。
1. 存储和访问前向传播计算所需的参数。
1. 根据需要初始化模型参数。


在下面的代码片段中，我们从零开始编写一个块。
它包含一个多层感知机，其具有256个隐藏单元的隐藏层和一个10维输出层。
注意，下面的`MLP`类继承了表示块的类。
我们的实现只需要提供我们自己的构造函数（Python中的`__init__`函数）和前向传播函数。


In [2]:
class MLP(nn.Module):
    # 用模型参数声明层。这里，我们声明两个全连接的层
    def __init__(self):
        # 调用MLP的父类Module的构造函数来执行必要的初始化。
        # 这样，在类实例化时也可以指定其他函数参数，例如模型参数params（稍后将介绍）
        super().__init__()
        self.hidden = nn.Linear(20, 256)  # 隐藏层
        self.out = nn.Linear(256, 10)  # 输出层

    # 定义模型的前向传播，即如何根据输入X返回所需的模型输出
    def forward(self, X):
        # 注意，这里我们使用ReLU的函数版本，其在nn.functional模块中定义。
        return self.out(F.relu(self.hidden(X)))

我们首先看一下前向传播函数，它以`X`作为输入，
计算带有激活函数的隐藏表示，并输出其未规范化的输出值。
在这个`MLP`实现中，两个层都是实例变量。
要了解这为什么是合理的，可以想象实例化两个多层感知机（`net1`和`net2`），
并根据不同的数据对它们进行训练。
当然，我们希望它们学到两种不同的模型。

接着我们[**实例化多层感知机的层，然后在每次调用前向传播函数时调用这些层**]。
注意一些关键细节：
首先，我们定制的`__init__`函数通过`super().__init__()`
调用父类的`__init__`函数，
省去了重复编写模版代码的痛苦。
然后，我们实例化两个全连接层，
分别为`self.hidden`和`self.out`。
注意，除非我们实现一个新的运算符，
否则我们不必担心反向传播函数或参数初始化，
系统将自动生成这些。

我们来试一下这个函数：


In [3]:
net = MLP()
net(X)

tensor([[ 0.1225,  0.0763,  0.1107, -0.1520,  0.1572,  0.1654, -0.0559,  0.1722,
         -0.1325,  0.0181],
        [ 0.1356,  0.0231,  0.0127, -0.0486,  0.0374,  0.1861,  0.1763,  0.2160,
         -0.1871, -0.0889]], grad_fn=<AddmmBackward0>)

块的一个主要优点是它的多功能性。
我们可以子类化块以创建层（如全连接层的类）、
整个模型（如上面的`MLP`类）或具有中等复杂度的各种组件。
我们在接下来的章节中充分利用了这种多功能性，
比如在处理卷积神经网络时。

## **顺序块**

现在我们可以更仔细地看看`Sequential`类是如何工作的，
回想一下`Sequential`的设计是为了把其他模块串起来。
为了构建我们自己的简化的`MySequential`，
我们只需要定义两个关键函数：

1. 一种将块逐个追加到列表中的函数；
1. 一种前向传播函数，用于将输入按追加块的顺序传递给块组成的“链条”。

下面的`MySequential`类提供了与默认`Sequential`类相同的功能。


In [5]:
class MySequential(nn.Module):
    def __init__(self, *args):
        super().__init__()
        for idx, module in enumerate(args):
            # 这里，module是Module子类的一个实例。我们把它保存在'Module'类的成员
            # 变量_modules中。_module的类型是OrderedDict
            self._modules[str(idx)] = module

    def forward(self, X):
        # OrderedDict保证了按照成员添加的顺序遍历它们
        for block in self._modules.values():
            X = block(X)
        return X

`__init__`函数将每个模块逐个添加到有序字典`_modules`中。
读者可能会好奇为什么每个`Module`都有一个`_modules`属性？
以及为什么我们使用它而不是自己定义一个Python列表？
简而言之，`_modules`的主要优点是：
在模块的参数初始化过程中，
系统知道在`_modules`字典中查找需要初始化参数的子块。


当`MySequential`的前向传播函数被调用时，
每个添加的块都按照它们被添加的顺序执行。
现在可以使用我们的`MySequential`类重新实现多层感知机。


In [8]:
net = MySequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))
net(X)

tensor([[ 0.0784, -0.1366,  0.1425,  0.1652, -0.0581, -0.2747, -0.0814, -0.0635,
         -0.0593, -0.1744],
        [ 0.1914, -0.1781,  0.0837,  0.1918, -0.0588, -0.0856,  0.0441, -0.0325,
          0.0026, -0.2751]], grad_fn=<AddmmBackward0>)

请注意，`MySequential`的用法与之前为`Sequential`类编写的代码相同
（如 :numref:`sec_mlp_concise` 中所述）。

## **在前向传播函数中执行代码**

`Sequential`类使模型构造变得简单，
允许我们组合新的架构，而不必定义自己的类。
然而，并不是所有的架构都是简单的顺序架构。
当需要更强的灵活性时，我们需要定义自己的块。
例如，我们可能希望在前向传播函数中执行Python的控制流。
此外，我们可能希望执行任意的数学运算，
而不是简单地依赖预定义的神经网络层。

到目前为止，
我们网络中的所有操作都对网络的激活值及网络的参数起作用。
然而，有时我们可能希望合并既不是上一层的结果也不是可更新参数的项，
我们称之为*常数参数*（constant parameter）。
例如，我们需要一个计算函数
$f(\mathbf{x},\mathbf{w}) = c \cdot \mathbf{w}^\top \mathbf{x}$的层，
其中$\mathbf{x}$是输入，
$\mathbf{w}$是参数，
$c$是某个在优化过程中没有更新的指定常量。
因此我们实现了一个`FixedHiddenMLP`类，如下所示：


In [9]:
class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # 不计算梯度的随机权重参数。因此其在训练期间保持不变
        self.rand_weight = torch.rand((20, 20), requires_grad=False)
        self.linear = nn.Linear(20, 20)

    def forward(self, X):
        X = self.linear(X)
        # 使用创建的常量参数以及relu和mm函数
        X = F.relu(torch.mm(X, self.rand_weight) + 1)
        # 复用全连接层。这相当于两个全连接层共享参数
        X = self.linear(X)
        # 控制流
        while X.abs().sum() > 1:
            X /= 2
        return X.sum()

在这个`FixedHiddenMLP`模型中，我们实现了一个隐藏层，
其权重（`self.rand_weight`）在实例化时被随机初始化，之后为常量。
这个权重不是一个模型参数，因此它永远不会被反向传播更新。
然后，神经网络将这个固定层的输出通过一个全连接层。

注意，在返回输出之前，模型做了一些不寻常的事情：
它运行了一个while循环，在$L_1$范数大于$1$的条件下，
将输出向量除以$2$，直到它满足条件为止。
最后，模型返回了`X`中所有项的和。
注意，此操作可能不会常用于在任何实际任务中，
我们只展示如何将任意代码集成到神经网络计算的流程中。


In [10]:
net = FixedHiddenMLP()
net(X)

tensor(-0.1077, grad_fn=<SumBackward0>)

我们可以**混合搭配各种组合块的方法**。
在下面的例子中，我们以一些想到的方法嵌套块。


In [12]:
class NestMLP(nn.Module):
    """定义嵌套的多层感知机（MLP）"""

    def __init__(self):
        super().__init__()  # 初始化父类

        # 定义包含两个隐藏层的网络块
        self.net = nn.Sequential(
            nn.Linear(20, 64),  # 第一层：20 → 64
            nn.ReLU(),          # ReLU激活函数
            nn.Linear(64, 32),  # 第二层：64 → 32
            nn.ReLU()           # ReLU激活函数
        )

        self.linear = nn.Linear(32, 16)  # 输出层：32 → 16

    def forward(self, X):
        """定义前向传播"""
        return self.linear(self.net(X))  # 先执行net，再执行linear


# 组合多个网络块，构建更复杂的神经网络
chimera = nn.Sequential(
    NestMLP(),         # 嵌套MLP：20 → 64 → 32 → 16
    nn.Linear(16, 20), # 全连接层：16 → 20
    FixedHiddenMLP()   # 自定义网络块（需要提前定义）
)

chimera(X)  # 执行前向传播

tensor(-0.2439, grad_fn=<SumBackward0>)

## 效率


读者可能会开始担心操作效率的问题。
毕竟，我们在一个高性能的深度学习库中进行了大量的字典查找、
代码执行和许多其他的Python代码。
Python的问题[全局解释器锁](https://wiki.python.org/moin/GlobalInterpreterLock)
是众所周知的。
在深度学习环境中，我们担心速度极快的GPU可能要等到CPU运行Python代码后才能运行另一个作业。


## 小结

* 一个块可以由许多层组成；一个块可以由许多块组成。
* 块可以包含代码。
* 块负责大量的内部处理，包括参数初始化和反向传播。
* 层和块的顺序连接由`Sequential`块处理。